In [59]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집II</b></font>
# 1절. selenium을 이용한 동적 웹크롤링 문법
- https://selenium-python.readthedocs.io/
- `pip install selenium` (아나콘다 프롬프트)
    - 경고 무시 => pip install --upgrade requests (requests를 최신버전으로 upgrade)하거나,
                 conda install urllib3==1.26.18
- selenium버전 : 4.47 / requests버전 : 2.28.1 / urllib3버전 :2.7.0

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

In [39]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
# By.CLASS_NAME, By.ID, By.CSS_SELECTOR, By.TAG_NAME
# a태그에서 By.LINK_TEXT, by.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN) # enter 

In [37]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
elem.send_keys(Keys.CONTROL, 'a') # ctrl+a
elem.send_keys('pycon')
btn_elem = dv.find_element(By.CSS_SELECTOR, 'button#submit')#GO버튼
btn_elem.click()

In [45]:
result_list = dv.find_elements(By.CSS_SELECTOR, 'li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.get_attribute('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [49]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - /psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - /events/python-events/378/
PyCon Australia 2013 - /events/python-events/57/


In [55]:
from urllib.parse import urlparse
current_url = dv.current_url
print('현재 url :', current_url)
result_parse = urlparse(current_url)
print('url parsing 결과 :', result_parse)
domain = f'{result_parse.scheme}://{result_parse.netloc}'
domain = "{}://{}".format(result_parse.scheme, result_parse.netloc)
print('현재 domain :', domain)

현재 url : https://www.python.org/search/?q=pycon&submit=
url parsing 결과 : ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain : https://www.python.org


In [56]:
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, 
                           domain+result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [ ]:
dv.close() # 브라우저 종료

# 2절. 동적웹크롤링 예제
## 2.1 다음 뉴스 검색

In [70]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

검색할 단어는?가을


In [81]:
bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
# len(bodies)
for body in bodies:
    a = body.find_element(By.TAG_NAME, 'a')
    title = a.text
    link  = a.get_attribute('href')
    # print(title, link)
    news_list.append([title, link])

In [80]:
page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
# page_nav.text
nex_page = page_nav.find_element(By.LINK_TEXT, "4") # a태그의 text가 2인 a태그
nex_page.click()


In [84]:
import pandas as pd
pd.DataFrame(news_list, columns=['뉴스제목','링크']).shape

(40, 2)

## 2-2 다음뉴스 페이징 처리
- 위의 예제를 이용하여 원하는 페이지만큼 뉴스 검색 결과를 받아오기

In [85]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

# query = 'AI'
query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

pages = int(input('몇 페이지 크롤링 할까요?'))
for page in range(1, pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
    for body in bodies:
        a = body.find_element(By.TAG_NAME, 'a')
        title = a.text
        link  = a.get_attribute('href')
        # print(title, link)
        news_list.append([title, link])
    page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
    nex_page = page_nav.find_element(By.LINK_TEXT, str(page+1)) # a태그의 text가 2인 a태그
    nex_page.click()
    time.sleep(2)
driver.close()
news_df = pd.DataFrame(news_list, columns=['title','link'])
display(news_df.head())
print(news_df.shape)

검색할 단어는?에이아이
몇 페이지 크롤링 할까요?3
1
2
3
